34 — F1 + F3 Final Acceptance / Freeze
Bu notebook yeni model eğitmez.

Amaç:

F1 sıcaklık OOD düzeltmesini aynı 31-hücre havuzu, sıfır train/test batarya çakışması, düzeltilmiş paired LOBO referansı ve bootstrap CI ile dondurmak.
F3 kanonik eğitim sözleşmesini exact artifact hash'leri, float32 X / float64 y, 20/20 tree reconstruction ve 34-batarya LOBO eşliği ile dondurmak.
Step 33 downstream acceptance zincirini zorunlu üst-kapı yapmak.
Eksik/değişmiş/stale dosyada HARD FAIL üretmek.
Sonraki paper/release aşamasının okuyabileceği tek bir F1_F3_acceptance_v1.json artefact'ı üretmek.
Yetkili protokol:

ONLINE_FIXED_START_CAUSAL_10S_V1

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import tempfile
import copy

import numpy as np
import pandas as pd

ROOT = Path(r"${PROJECT_ROOT}")
RESULTS = ROOT / "results"

F1_DIR = RESULTS / "26_F1_temperature_matched_pool_ood_fix"
F3_DIR = RESULTS / "27_F3_canonical_target_environment_contract"
STEP33_DIR = RESULTS / "33_downstream_hard_fail_acceptance_v3"

STEP33_ACCEPTANCE = STEP33_DIR / "downstream_acceptance_v3.json"

OUT = RESULTS / "34_F1_F3_final_acceptance_freeze"
OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "ONLINE_FIXED_START_CAUSAL_10S_V1"

EXPECTED_MIXED_EXCLUDED = {"B0038", "B0039", "B0040"}
EXPECTED_ALLOWED_COUNT = 31
EXPECTED_FULL_LOBO_BATTERIES = 34
EXPECTED_ROWS = 2521
EXPECTED_FEATURES = 44

# ------------------------------------------------------------------
# Frozen critical artifact hashes from the 34A inventory.
# ------------------------------------------------------------------

EXPECTED_F1_HASHES = {
    "F1_independent_audit_value_sanity.csv":
        "7e3724bc5eff84a27c2237551f2cf30b3919a2b5b19e4480402f88bdd72a15a2",
    "F1_temperature_claim_registry_patch.csv":
        "969319e0ad59dcc2eeee6b4ad45fe7e0ba65a69a3c5880d09fbe94b77b00834d",
    "F1_temperature_matched_pool_metadata.json":
        "0d1b28c932dc3ee6d215c441f6405b2c150672fb8e068bbbf9721ffe359c4d35",
    "temperature_allowed_31_batteries.csv":
        "0f83bcd3bf88433cf2d34822a13526426049c201140bc08ae017e085154712d1",
    "temperature_condition_fold_train_test_ids.csv":
        "db35409df68d023dad7b207edfc3c572f7362ab131ecb37e8c804adaa7ce57ec",
    "temperature_corrected_aggregate.csv":
        "47b829ed62971fddc5db1b54f62d60fe6e80d097593532f65207dc987b302e36",
    "temperature_corrected_paired_by_battery.csv":
        "b4257a32b93147fb7179e124d749f4cd5a42eeaf992d6d06833a344ec3bd8445",
    "temperature_corrected_penalty_bootstrap.csv":
        "c0e9d7a4ed16f998974a70ad9626ddf443e8eb9ab34eb83275ea2c22e43397c1",
    "temperature_matched_31cell_lobo_by_battery.csv":
        "844e31ae8c01db3d3163eb60806d044f53f0725da2931afbb22e9ce2614f3c4b",
    "temperature_matched_31cell_lobo_fold_train_test_ids.csv":
        "b5d4e13fee93d5ff0cada3cfe3e8682368855b94dea1db762156ac56fcec2f69",
    "temperature_old_vs_corrected.csv":
        "c373e5f0d0f124145a50b219434dc158611f2048a4cf70638a0b910d3efee454",
}

EXPECTED_F3_HASHES = {
    "canonical_float64_seed42_lobo_by_battery.csv":
        "8beb460700a6652d7099dba8af47182b824797393635c5cc6c292cafcbee7258",
    "canonical_float64_tree_exact_match.csv":
        "03ac21b221db73b07d5c0e1579357c12052b335e67ae5a2a0e18482411605cc3",
    "canonical_lobo_vs_frozen_comparison.csv":
        "c70c5a07985a2f05060e47c33c62cf374c6243459241db784d2785f312c82867",
    "canonical_training_environment.json":
        "cffbb78d32feea7fa00ece67b35f48a96a9011787c8ba46bb7d3a90a001eae21",
    "environment_repro_exact.yml":
        "f8cd7bc3ae454068a1d0278cc16e0d0aaa4a5cd8f006581b77340093dd728ecb",
    "F3_artifact_manifest_sha256.csv":
        "55a85e6f2a7a1d42e3bbac140a759fe15f35657e1c82891a22872231c3d2f1c0",
    "final_online_training_canonical_v2.npz":
        "4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962",
    "legacy_float32_target_tree_match.csv":
        "e36e7f5e7658faf942e46503cf8d81ce191b10044710f97febe1bd22534c6033",
    "README_CANONICAL_TRAINING_CONTRACT.txt":
        "30c45f9be77fa4db71f5717cc7428e9db455b32796704166c8d6028a0c744766",
    "requirements_repro_exact.txt":
        "f33ba8c4f2871857cf7393f97e844929caa529150f679cb802228a97aa35317a",
    "rf20x7_parameter_check.csv":
        "b46e4f86658d6a255cf4a3a5876863251653a6fc99e54b9d95c5223cbb5bbff3",
    "target_precision_audit.csv":
        "0c8135ee99500531290d5c146a7a825daa565be6f22aaeaf64c9d32d7ce3ab5e",
}

for p in [F1_DIR, F3_DIR, STEP33_ACCEPTANCE]:
    if not p.exists():
        raise FileNotFoundError(p)

print("All required top-level paths exist.")


All required top-level paths exist.


In [2]:

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(1024 * 1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def assert_close(name, observed, expected, atol=1e-6):
    observed = float(observed)
    expected = float(expected)
    if not np.isfinite(observed):
        raise RuntimeError(f"{name} is non-finite.")
    if abs(observed - expected) > atol:
        raise RuntimeError(
            f"{name} mismatch: observed={observed}, expected={expected}, atol={atol}"
        )

def as_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in {"true", "1", "yes", "pass"}

def validate_step33(path):
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        raise RuntimeError("Step 33 acceptance is missing or empty.")

    d = json.loads(p.read_text(encoding="utf-8"))

    if d.get("schema_version") != "DOWNSTREAM_ACCEPTANCE_V3":
        raise RuntimeError("Unexpected Step 33 schema.")
    if d.get("protocol_id") != PROTOCOL_ID:
        raise RuntimeError("Step 33 protocol mismatch.")
    if d.get("DOWNSTREAM_FINAL_ACCEPTANCE") is not True:
        raise RuntimeError("Step 33 is not accepted.")

    checks = d.get("checks")
    if not isinstance(checks, list) or not checks:
        raise RuntimeError("Step 33 checks missing/empty.")
    if not all(as_bool(x.get("pass")) for x in checks):
        raise RuntimeError("Step 33 contains a failed check.")

    return d

step33 = validate_step33(STEP33_ACCEPTANCE)

print("Step 33 upstream gate PASS.")
print("Step 33 SHA256:", sha256_file(STEP33_ACCEPTANCE))


Step 33 upstream gate PASS.
Step 33 SHA256: a97dfde54359d9ecb5d5d2348429a6f25b0bb0f1c9b54ea29a74c414fd77e162


In [3]:

hash_rows = []

def validate_hash_set(base, expected_map, group):
    for name, expected in expected_map.items():
        p = base / name

        if not p.exists():
            raise RuntimeError(f"{group} required artifact missing: {name}")

        actual = sha256_file(p)
        ok = actual == expected

        hash_rows.append({
            "group": group,
            "filename": name,
            "expected_sha256": expected,
            "actual_sha256": actual,
            "pass": ok,
        })

        if not ok:
            raise RuntimeError(
                f"{group} SHA256 mismatch for {name}\n"
                f"Observed: {actual}\nExpected: {expected}"
            )

validate_hash_set(F1_DIR, EXPECTED_F1_HASHES, "F1")
validate_hash_set(F3_DIR, EXPECTED_F3_HASHES, "F3")

hash_df = pd.DataFrame(hash_rows)
display(hash_df)

hash_df.to_csv(
    OUT / "F1_F3_frozen_artifact_hashes.csv",
    index=False,
    encoding="utf-8-sig",
)

print("All frozen critical artifact hashes PASS.")


,group,filename,expected_sha256,actual_sha256,pass
0,F1,F1_independent_audit_value_sanity.csv,7e3724bc5eff84a27c2237551f2cf30b3919a2b5b19e44...,7e3724bc5eff84a27c2237551f2cf30b3919a2b5b19e44...,True
1,F1,F1_temperature_claim_registry_patch.csv,969319e0ad59dcc2eeee6b4ad45fe7e0ba65a69a3c5880...,969319e0ad59dcc2eeee6b4ad45fe7e0ba65a69a3c5880...,True
2,F1,F1_temperature_matched_pool_metadata.json,0d1b28c932dc3ee6d215c441f6405b2c150672fb8e068b...,0d1b28c932dc3ee6d215c441f6405b2c150672fb8e068b...,True
3,F1,temperature_allowed_31_batteries.csv,0f83bcd3bf88433cf2d34822a13526426049c201140bc0...,0f83bcd3bf88433cf2d34822a13526426049c201140bc0...,True
4,F1,temperature_condition_fold_train_test_ids.csv,db35409df68d023dad7b207edfc3c572f7362ab131ecb3...,db35409df68d023dad7b207edfc3c572f7362ab131ecb3...,True
5,F1,temperature_corrected_aggregate.csv,47b829ed62971fddc5db1b54f62d60fe6e80d097593532...,47b829ed62971fddc5db1b54f62d60fe6e80d097593532...,True
6,F1,temperature_corrected_paired_by_battery.csv,b4257a32b93147fb7179e124d749f4cd5a42eeaf992d6d...,b4257a32b93147fb7179e124d749f4cd5a42eeaf992d6d...,True
7,F1,temperature_corrected_penalty_bootstrap.csv,c0e9d7a4ed16f998974a70ad9626ddf443e8eb9ab34eb8...,c0e9d7a4ed16f998974a70ad9626ddf443e8eb9ab34eb8...,True
8,F1,temperature_matched_31cell_lobo_by_battery.csv,844e31ae8c01db3d3163eb60806d044f53f0725da2931a...,844e31ae8c01db3d3163eb60806d044f53f0725da2931a...,True
9,F1,temperature_matched_31cell_lobo_fold_train_tes...,b5d4e13fee93d5ff0cada3cfe3e8682368855b94dea1db...,b5d4e13fee93d5ff0cada3cfe3e8682368855b94dea1db...,True


All frozen critical artifact hashes PASS.


In [4]:

allowed = pd.read_csv(F1_DIR / "temperature_allowed_31_batteries.csv")

if list(allowed.columns) != ["battery_id"]:
    raise RuntimeError("Unexpected allowed-pool schema.")

allowed_ids = allowed["battery_id"].astype(str).tolist()
allowed_set = set(allowed_ids)

if len(allowed) != EXPECTED_ALLOWED_COUNT:
    raise RuntimeError("Allowed temperature pool is not 31 rows.")
if len(allowed_set) != EXPECTED_ALLOWED_COUNT:
    raise RuntimeError("Allowed temperature pool IDs are not unique.")
if allowed_set & EXPECTED_MIXED_EXCLUDED:
    raise RuntimeError("Mixed-temperature excluded cells leaked into allowed pool.")

print("Allowed pool count:", len(allowed_set))
print("Excluded mixed IDs absent:", sorted(EXPECTED_MIXED_EXCLUDED))


Allowed pool count: 31
Excluded mixed IDs absent: ['B0038', 'B0039', 'B0040']


In [5]:

condition_folds = pd.read_csv(
    F1_DIR / "temperature_condition_fold_train_test_ids.csv"
)

matched_folds = pd.read_csv(
    F1_DIR / "temperature_matched_31cell_lobo_fold_train_test_ids.csv"
)

# Temperature condition holdout: 3 families, no train/test battery overlap.
if len(condition_folds) != 3:
    raise RuntimeError("Expected exactly 3 temperature-family holdout rows.")

if "battery_overlap" not in condition_folds.columns:
    raise RuntimeError("Condition-fold table lacks battery_overlap.")
if not (pd.to_numeric(condition_folds["battery_overlap"]) == 0).all():
    raise RuntimeError("Battery overlap detected in temperature condition holdout.")

# Matched strict LOBO: 31 batteries × 2 models = 62 folds.
if len(matched_folds) != 62:
    raise RuntimeError("Matched LOBO fold table must contain 62 rows.")

required_cols = {
    "model", "test_battery", "allowed_pool_count",
    "allowed_pool_ids", "excluded_mixed_ids", "battery_overlap"
}
if not required_cols.issubset(set(matched_folds.columns)):
    raise RuntimeError("Matched LOBO fold table missing required columns.")

if not (pd.to_numeric(matched_folds["allowed_pool_count"]) == 31).all():
    raise RuntimeError("Matched LOBO allowed_pool_count is not 31 everywhere.")

if not (pd.to_numeric(matched_folds["battery_overlap"]) == 0).all():
    raise RuntimeError("Battery overlap detected in matched LOBO.")

models = sorted(matched_folds["model"].astype(str).unique())
expected_models = ["RF_20x7_deployed", "RF_full_150"]

if models != expected_models:
    raise RuntimeError(f"Unexpected F1 models: {models}")

for model in expected_models:
    m = matched_folds[matched_folds["model"] == model]

    if len(m) != 31:
        raise RuntimeError(f"{model}: expected 31 LOBO folds.")
    if m["test_battery"].nunique() != 31:
        raise RuntimeError(f"{model}: test batteries are not 31 unique cells.")
    if set(m["test_battery"].astype(str)) != allowed_set:
        raise RuntimeError(f"{model}: test batteries do not equal allowed 31-cell pool.")

# Every fold must list exactly the same allowed pool and same excluded mixed cells.
for _, row in matched_folds.iterrows():
    row_allowed = set(str(row["allowed_pool_ids"]).split("|"))
    row_excluded = set(str(row["excluded_mixed_ids"]).split("|"))

    if row_allowed != allowed_set:
        raise RuntimeError("A matched LOBO fold has a mismatched allowed_pool_ids set.")
    if row_excluded != EXPECTED_MIXED_EXCLUDED:
        raise RuntimeError("A matched LOBO fold has mismatched excluded_mixed_ids.")

print("F1 fold isolation PASS.")
print("Condition holdout battery overlap: 0 for all 3 rows.")
print("Matched LOBO: 31 batteries × 2 models, overlap 0.")


F1 fold isolation PASS.
Condition holdout battery overlap: 0 for all 3 rows.
Matched LOBO: 31 batteries × 2 models, overlap 0.


In [6]:

agg = pd.read_csv(F1_DIR / "temperature_corrected_aggregate.csv")

if len(agg) != 2:
    raise RuntimeError("Corrected aggregate must have 2 rows.")

if not (agg["protocol_id"].astype(str) == PROTOCOL_ID).all():
    raise RuntimeError("F1 aggregate protocol mismatch.")

if not (pd.to_numeric(agg["batteries"]) == 31).all():
    raise RuntimeError("F1 aggregate does not use 31 batteries.")

expected_agg = {
    "RF_20x7_deployed": {
        "condition_battery_macro_mae_pp": 13.99005195904802,
        "matched_lobo_battery_macro_mae_pp": 6.993527195938773,
        "mean_matched_temperature_penalty_pp": 6.996524763109244,
    },
    "RF_full_150": {
        "condition_battery_macro_mae_pp": 13.062143313385874,
        "matched_lobo_battery_macro_mae_pp": 7.063627836126513,
        "mean_matched_temperature_penalty_pp": 5.998515477259362,
    },
}

for model, vals in expected_agg.items():
    row = agg[agg["model"] == model]
    if len(row) != 1:
        raise RuntimeError(f"Missing/duplicate aggregate row for {model}.")
    row = row.iloc[0]

    for col, expected in vals.items():
        assert_close(
            f"{model}/{col}",
            row[col],
            expected,
            atol=1e-9,
        )

print("F1 corrected aggregate values PASS.")


F1 corrected aggregate values PASS.


In [7]:

boot = pd.read_csv(
    F1_DIR / "temperature_corrected_penalty_bootstrap.csv"
)

if len(boot) != 2:
    raise RuntimeError("Temperature bootstrap table must have 2 rows.")

if not (boot["protocol_id"].astype(str) == PROTOCOL_ID).all():
    raise RuntimeError("Bootstrap protocol mismatch.")

if not (pd.to_numeric(boot["batteries"]) == 31).all():
    raise RuntimeError("Bootstrap battery count mismatch.")

if not (pd.to_numeric(boot["bootstrap_repeats"]) == 5000).all():
    raise RuntimeError("Bootstrap repeat count is not 5000.")

if boot["model_retrained_inside_bootstrap"].map(as_bool).any():
    raise RuntimeError("Bootstrap incorrectly claims model retraining inside bootstrap.")

expected_boot = {
    "RF_20x7_deployed": (6.996524763109244, 4.1168398024248685, 9.712288932257596),
    "RF_full_150": (5.998515477259362, 3.2768166753665358, 8.789110507712824),
}

for model, (mean_, lo, hi) in expected_boot.items():
    row = boot[boot["model"] == model]
    if len(row) != 1:
        raise RuntimeError(f"Missing/duplicate bootstrap row for {model}.")
    row = row.iloc[0]

    assert_close(f"{model}/bootstrap_mean", row["mean_matched_temperature_penalty_pp"], mean_, 1e-9)
    assert_close(f"{model}/ci_low", row["ci95_low_pp"], lo, 1e-6)
    assert_close(f"{model}/ci_high", row["ci95_high_pp"], hi, 1e-6)

audit = pd.read_csv(F1_DIR / "F1_independent_audit_value_sanity.csv")

if len(audit) != 6:
    raise RuntimeError("F1 audit sanity table must contain 6 rows.")

if not audit["pass_tol_5e-4_pp"].map(as_bool).all():
    raise RuntimeError("F1 independent audit sanity contains a failed row.")

claims = pd.read_csv(F1_DIR / "F1_temperature_claim_registry_patch.csv")

if len(claims) != 6:
    raise RuntimeError("F1 claim registry must contain 6 rows.")

if not (claims["protocol_id"].astype(str) == PROTOCOL_ID).all():
    raise RuntimeError("F1 claim registry protocol mismatch.")

print("F1 bootstrap + audit + claim registry PASS.")


F1 bootstrap + audit + claim registry PASS.


In [8]:

npz_path = F3_DIR / "final_online_training_canonical_v2.npz"
z = np.load(npz_path, allow_pickle=False)

required_arrays = {
    "X_features",
    "y_soh_rated_pct",
    "battery_id",
    "cycle_uid",
}

if set(z.files) != required_arrays:
    raise RuntimeError(f"Unexpected canonical NPZ arrays: {z.files}")

X = z["X_features"]
y = z["y_soh_rated_pct"]
battery_id = z["battery_id"]
cycle_uid = z["cycle_uid"]

npz_checks = {
    "X_shape": X.shape == (2521, 44),
    "X_dtype_float32": X.dtype == np.float32,
    "y_shape": y.shape == (2521,),
    "y_dtype_float64": y.dtype == np.float64,
    "battery_shape": battery_id.shape == (2521,),
    "cycle_uid_shape": cycle_uid.shape == (2521,),
    "battery_count_34": len(np.unique(battery_id)) == 34,
    "cycle_uid_unique_2521": len(np.unique(cycle_uid)) == 2521,
}

if not all(npz_checks.values()):
    raise RuntimeError(f"Canonical NPZ contract failed: {npz_checks}")

print("Canonical NPZ contract PASS.")
print(npz_checks)


Canonical NPZ contract PASS.
{'X_shape': True, 'X_dtype_float32': True, 'y_shape': True, 'y_dtype_float64': True, 'battery_shape': True, 'cycle_uid_shape': True, 'battery_count_34': True, 'cycle_uid_unique_2521': True}


In [9]:

trees = pd.read_csv(
    F3_DIR / "canonical_float64_tree_exact_match.csv"
)

if len(trees) != 20:
    raise RuntimeError("Canonical tree table must contain 20 trees.")

if not trees["exact_tree_match"].map(as_bool).all():
    raise RuntimeError("Canonical tree exact reconstruction is not 20/20.")

legacy = pd.read_csv(
    F3_DIR / "legacy_float32_target_tree_match.csv"
)

if len(legacy) != 20:
    raise RuntimeError("Legacy tree table must contain 20 trees.")

if legacy["exact_tree_match"].map(as_bool).any():
    raise RuntimeError(
        "Legacy float32 target unexpectedly reconstructs one or more final trees."
    )

if pd.to_numeric(trees["saved_nodes"]).sum() != pd.to_numeric(trees["reconstructed_nodes"]).sum():
    raise RuntimeError("Canonical total reconstructed node count mismatch.")

print("Canonical tree matches: 20/20")
print("Legacy float32 target matches: 0/20")
print("Total canonical nodes:", int(pd.to_numeric(trees["saved_nodes"]).sum()))


Canonical tree matches: 20/20
Legacy float32 target matches: 0/20
Total canonical nodes: 3968


In [10]:

lobo = pd.read_csv(
    F3_DIR / "canonical_float64_seed42_lobo_by_battery.csv"
)

comp = pd.read_csv(
    F3_DIR / "canonical_lobo_vs_frozen_comparison.csv"
)

if len(lobo) != 34 or lobo["test_battery"].nunique() != 34:
    raise RuntimeError("Canonical LOBO does not contain 34 unique test batteries.")

if len(comp) != 34 or comp["test_battery"].nunique() != 34:
    raise RuntimeError("LOBO-vs-frozen comparison does not contain 34 unique batteries.")

macro_mae = float(pd.to_numeric(lobo["mae_pp"]).mean())
max_abs_diff = float(pd.to_numeric(comp["abs_mae_diff_pp"]).max())

assert_close(
    "canonical_LOBO_macro_MAE",
    macro_mae,
    7.096415349426006,
    atol=1e-12,
)

if max_abs_diff > 3e-14:
    raise RuntimeError(
        f"Canonical-vs-frozen per-battery MAE diff too large: {max_abs_diff}"
    )

print("Canonical 34-battery LOBO macro MAE:", macro_mae)
print("Max per-battery MAE diff vs frozen:", max_abs_diff)


Canonical 34-battery LOBO macro MAE: 7.096415349426007
Max per-battery MAE diff vs frozen: 2.3092638912203256e-14


In [11]:

params = pd.read_csv(F3_DIR / "rf20x7_parameter_check.csv")

if len(params) != 5:
    raise RuntimeError("RF20x7 parameter table must contain 5 rows.")
if not params["pass"].map(as_bool).all():
    raise RuntimeError("RF20x7 parameter check contains a failure.")

precision = pd.read_csv(F3_DIR / "target_precision_audit.csv")

if len(precision) != 1:
    raise RuntimeError("Target precision audit must contain 1 row.")

r = precision.iloc[0]

if int(r["rows"]) != 2521 or int(r["features"]) != 44:
    raise RuntimeError("Target precision audit row/feature count mismatch.")
if str(r["X_dtype"]) != "float32":
    raise RuntimeError("Canonical X dtype is not float32.")
if str(r["canonical_y_dtype"]) != "float64":
    raise RuntimeError("Canonical y dtype is not float64.")

target_diff = float(r["max_abs_target_diff_pp"])
if target_diff > 5e-6:
    raise RuntimeError(
        f"Legacy-vs-canonical target precision difference too large: {target_diff}"
    )

requirements = (
    F3_DIR / "requirements_repro_exact.txt"
).read_text(encoding="utf-8").strip().splitlines()

required_versions = {
    "numpy==2.1.3",
    "pandas==2.2.3",
    "scikit-learn==1.6.1",
    "joblib==1.4.2",
    "scipy==1.15.3",
    "matplotlib==3.10.0",
    "nbformat==5.10.4",
}

if set(x.strip() for x in requirements if x.strip()) != required_versions:
    raise RuntimeError("Exact reproducibility requirements mismatch.")

env_text = (
    F3_DIR / "canonical_training_environment.json"
).read_text(encoding="utf-8")

for token in [
    PROTOCOL_ID,
    "1.6.1",
    "7.096415349426",
    "4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962",
]:
    if token not in env_text:
        raise RuntimeError(
            f"Canonical environment JSON missing expected token: {token}"
        )

print("F3 parameters + precision + environment PASS.")


F3 parameters + precision + environment PASS.


In [12]:

manifest = pd.read_csv(
    F3_DIR / "F3_artifact_manifest_sha256.csv"
)

required_manifest_cols = {"filename", "size_bytes", "sha256"}
if not required_manifest_cols.issubset(manifest.columns):
    raise RuntimeError("F3 manifest schema mismatch.")

manifest_results = []

for _, row in manifest.iterrows():
    p = F3_DIR / str(row["filename"])

    if not p.exists():
        raise RuntimeError(
            f"F3 manifest references missing file: {row['filename']}"
        )

    actual_size = p.stat().st_size
    actual_sha = sha256_file(p)

    ok = (
        actual_size == int(row["size_bytes"])
        and actual_sha == str(row["sha256"])
    )

    manifest_results.append({
        "filename": row["filename"],
        "size_match": actual_size == int(row["size_bytes"]),
        "sha_match": actual_sha == str(row["sha256"]),
        "pass": ok,
    })

    if not ok:
        raise RuntimeError(
            f"F3 manifest verification failed: {row['filename']}"
        )

manifest_check_df = pd.DataFrame(manifest_results)

display(manifest_check_df)

manifest_check_df.to_csv(
    OUT / "F3_manifest_verification.csv",
    index=False,
    encoding="utf-8-sig",
)

print("F3 internal artifact manifest PASS.")


,filename,size_match,sha_match,pass
0,canonical_float64_seed42_lobo_by_battery.csv,True,True,True
1,canonical_float64_tree_exact_match.csv,True,True,True
2,canonical_lobo_vs_frozen_comparison.csv,True,True,True
3,canonical_training_environment.json,True,True,True
4,environment_repro_exact.yml,True,True,True
5,final_online_training_canonical_v2.npz,True,True,True
6,legacy_float32_target_tree_match.csv,True,True,True
7,README_CANONICAL_TRAINING_CONTRACT.txt,True,True,True
8,requirements_repro_exact.txt,True,True,True
9,rf20x7_parameter_check.csv,True,True,True


F3 internal artifact manifest PASS.


In [13]:

checks = [
    {"check_id": "F1_01", "check": "Critical F1 artifact hashes frozen", "pass": True},
    {"check_id": "F1_02", "check": "Temperature matched pool = 31 unique cells", "pass": True},
    {"check_id": "F1_03", "check": "B0038/B0039/B0040 excluded from strict pool", "pass": True},
    {"check_id": "F1_04", "check": "Temperature condition folds have zero battery overlap", "pass": True},
    {"check_id": "F1_05", "check": "Matched LOBO has 31 folds/model and zero overlap", "pass": True},
    {"check_id": "F1_06", "check": "Corrected aggregate values match frozen results", "pass": True},
    {"check_id": "F1_07", "check": "5000-repeat battery bootstrap CIs match frozen results", "pass": True},
    {"check_id": "F1_08", "check": "Independent audit sanity and claim registry pass", "pass": True},

    {"check_id": "F3_01", "check": "Critical F3 artifact hashes frozen", "pass": True},
    {"check_id": "F3_02", "check": "Canonical NPZ is 2521x44 float32 X + float64 y", "pass": True},
    {"check_id": "F3_03", "check": "Canonical target reconstructs 20/20 trees", "pass": True},
    {"check_id": "F3_04", "check": "Legacy float32 target reconstructs 0/20 trees", "pass": True},
    {"check_id": "F3_05", "check": "Canonical strict LOBO has 34 unique batteries", "pass": True},
    {"check_id": "F3_06", "check": "Canonical LOBO macro MAE = 7.096415349426 pp", "pass": True},
    {"check_id": "F3_07", "check": "Max per-battery MAE diff vs frozen <=3e-14 pp", "pass": True},
    {"check_id": "F3_08", "check": "RF20x7 parameter contract passes", "pass": True},
    {"check_id": "F3_09", "check": "Target precision contract passes", "pass": True},
    {"check_id": "F3_10", "check": "Exact software environment contract passes", "pass": True},
    {"check_id": "F3_11", "check": "F3 internal artifact manifest verifies", "pass": True},
]

all_pass = all(bool(x["pass"]) for x in checks)

acceptance = {
    "schema_version": "F1_F3_ACCEPTANCE_V1",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "protocol_id": PROTOCOL_ID,
    "upstream_step33_sha256": sha256_file(STEP33_ACCEPTANCE),
    "F1": {
        "allowed_batteries": 31,
        "excluded_mixed_batteries": sorted(EXPECTED_MIXED_EXCLUDED),
        "RF20_condition_mae_pp": 13.99005195904802,
        "RF20_matched_lobo_mae_pp": 6.993527195938773,
        "RF20_temperature_penalty_pp": 6.996524763109244,
        "RF20_temperature_penalty_ci95_pp": [
            4.1168398024248685,
            9.712288932257596,
        ],
        "RF150_condition_mae_pp": 13.062143313385874,
        "RF150_matched_lobo_mae_pp": 7.063627836126513,
        "RF150_temperature_penalty_pp": 5.998515477259362,
        "RF150_temperature_penalty_ci95_pp": [
            3.2768166753665358,
            8.789110507712824,
        ],
        "bootstrap_repeats": 5000,
        "bootstrap_retrains_model": False,
    },
    "F3": {
        "canonical_npz_sha256":
            "4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962",
        "rows": 2521,
        "features": 44,
        "X_dtype": "float32",
        "y_dtype": "float64",
        "batteries": 34,
        "tree_matches": "20/20",
        "legacy_float32_target_tree_matches": "0/20",
        "strict_lobo_macro_mae_pp": macro_mae,
        "max_battery_mae_diff_vs_frozen_pp": max_abs_diff,
        "sklearn_version": "1.6.1",
    },
    "checks": checks,
    "F1_F3_FINAL_ACCEPTANCE": all_pass,
}

ACCEPTANCE_JSON = OUT / "F1_F3_acceptance_v1.json"
ACCEPTANCE_JSON.write_text(
    json.dumps(acceptance, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

pd.DataFrame(checks).to_csv(
    OUT / "F1_F3_acceptance_checks.csv",
    index=False,
    encoding="utf-8-sig",
)

print("F1_F3_FINAL_ACCEPTANCE:", all_pass)


F1_F3_FINAL_ACCEPTANCE: True


In [14]:

def validate_f1_f3_acceptance(path):
    p = Path(path)

    if not p.exists():
        raise RuntimeError("F1/F3 acceptance missing.")
    if p.stat().st_size == 0:
        raise RuntimeError("F1/F3 acceptance empty.")

    d = json.loads(p.read_text(encoding="utf-8"))

    if d.get("schema_version") != "F1_F3_ACCEPTANCE_V1":
        raise RuntimeError("Wrong F1/F3 acceptance schema.")
    if d.get("protocol_id") != PROTOCOL_ID:
        raise RuntimeError("Wrong F1/F3 protocol.")
    if d.get("F1_F3_FINAL_ACCEPTANCE") is not True:
        raise RuntimeError("F1/F3 final acceptance is not True.")

    checks_ = d.get("checks")
    if not isinstance(checks_, list) or not checks_:
        raise RuntimeError("F1/F3 checks missing/empty.")
    if not all(as_bool(x.get("pass")) for x in checks_):
        raise RuntimeError("F1/F3 acceptance contains failed checks.")

    return d

validated = validate_f1_f3_acceptance(ACCEPTANCE_JSON)

def expect_reject(label, mutator=None, missing=False, empty=False):
    with tempfile.TemporaryDirectory() as td:
        p = Path(td) / "a.json"

        if missing:
            pass
        elif empty:
            p.write_text("", encoding="utf-8")
        else:
            d = json.loads(
                ACCEPTANCE_JSON.read_text(encoding="utf-8")
            )
            if mutator is not None:
                mutator(d)
            p.write_text(
                json.dumps(d, indent=2),
                encoding="utf-8",
            )

        rejected = False
        try:
            validate_f1_f3_acceptance(p)
        except RuntimeError:
            rejected = True

        return {
            "negative_gate": label,
            "rejected_as_required": rejected,
        }

negative = [
    expect_reject("missing acceptance", missing=True),
    expect_reject("empty acceptance", empty=True),
    expect_reject(
        "wrong protocol",
        lambda d: d.__setitem__("protocol_id", "WRONG_PROTOCOL"),
    ),
    expect_reject(
        "final acceptance false",
        lambda d: d.__setitem__("F1_F3_FINAL_ACCEPTANCE", False),
    ),
]

def fail_one(d):
    d["checks"][0]["pass"] = False

def empty_checks(d):
    d["checks"] = []

negative.append(expect_reject("one failed check", fail_one))
negative.append(expect_reject("empty checks", empty_checks))

neg_df = pd.DataFrame(negative)
display(neg_df)

negative_pass = bool(neg_df["rejected_as_required"].all())

neg_df.to_csv(
    OUT / "F1_F3_negative_gate_tests.csv",
    index=False,
    encoding="utf-8-sig",
)

print("F1/F3 negative gate tests PASS:", negative_pass)


,negative_gate,rejected_as_required
0,missing acceptance,True
1,empty acceptance,True
2,wrong protocol,True
3,final acceptance false,True
4,one failed check,True
5,empty checks,True


F1/F3 negative gate tests PASS: True


In [15]:

STEP34_PASS = bool(
    all_pass
    and negative_pass
)

print("=" * 104)
print("34 F1 + F3 FINAL ACCEPTANCE / FREEZE — FINAL SUMMARY")
print("=" * 104)

print("Upstream Step 33 validated:", True)

print()
print("F1 TEMPERATURE OOD")
print("Allowed strict pool:", len(allowed_set))
print("Excluded mixed cells:", "|".join(sorted(EXPECTED_MIXED_EXCLUDED)))
print("Condition-holdout battery overlap:", int(pd.to_numeric(condition_folds["battery_overlap"]).max()))
print("Matched-LOBO battery overlap:", int(pd.to_numeric(matched_folds["battery_overlap"]).max()))

print(
    "RF20 condition / matched LOBO / penalty:",
    13.99005195904802,
    6.993527195938773,
    6.996524763109244,
)
print(
    "RF20 penalty 95% CI:",
    4.1168398024248685,
    9.712288932257596,
)
print(
    "RF150 condition / matched LOBO / penalty:",
    13.062143313385874,
    7.063627836126513,
    5.998515477259362,
)
print(
    "RF150 penalty 95% CI:",
    3.2768166753665358,
    8.789110507712824,
)
print("Bootstrap repeats:", 5000)
print("Model retrained inside bootstrap:", False)

print()
print("F3 CANONICAL CONTRACT")
print("Canonical NPZ SHA256:", sha256_file(npz_path))
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Unique batteries:", len(np.unique(battery_id)))
print("Unique cycle_uid:", len(np.unique(cycle_uid)))
print("Canonical tree matches: 20/20")
print("Legacy float32 target matches: 0/20")
print("Canonical LOBO macro MAE pp:", macro_mae)
print("Max battery MAE diff vs frozen pp:", max_abs_diff)
print("scikit-learn:", "1.6.1")

print()
print("Critical frozen artifact hashes:", len(hash_df), "/", len(hash_df))
print("F3 manifest verification:", True)
print("F1/F3 hard-fail negative gates:", negative_pass)

print()
print("F1_F3_FINAL_ACCEPTANCE:", all_pass)
print("STEP 34 STATUS:", "PASS" if STEP34_PASS else "FAIL")

if not STEP34_PASS:
    raise RuntimeError(
        "STEP 34 failed. Do not proceed to Paper V3 / Freeze V3 packaging."
    )


34 F1 + F3 FINAL ACCEPTANCE / FREEZE — FINAL SUMMARY
Upstream Step 33 validated: True

F1 TEMPERATURE OOD
Allowed strict pool: 31
Excluded mixed cells: B0038|B0039|B0040
Condition-holdout battery overlap: 0
Matched-LOBO battery overlap: 0
RF20 condition / matched LOBO / penalty: 13.99005195904802 6.993527195938773 6.996524763109244
RF20 penalty 95% CI: 4.1168398024248685 9.712288932257596
RF150 condition / matched LOBO / penalty: 13.062143313385874 7.063627836126513 5.998515477259362
RF150 penalty 95% CI: 3.2768166753665358 8.789110507712824
Bootstrap repeats: 5000
Model retrained inside bootstrap: False

F3 CANONICAL CONTRACT
Canonical NPZ SHA256: 4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962
X: (2521, 44) float32
y: (2521,) float64
Unique batteries: 34
Unique cycle_uid: 2521
Canonical tree matches: 20/20
Legacy float32 target matches: 0/20
Canonical LOBO macro MAE pp: 7.096415349426007
Max battery MAE diff vs frozen pp: 2.3092638912203256e-14
scikit-learn: 1.6.1

C